## Required Python Libraries for the Raman Analysis Scripts

The Raman workflow uses two main Python scripts:

1. **Step 1 : Raman preprocessing**
   This script reads raw Raman spectra, applies baseline removal, smoothing, SNR calculation, saves processed `.parquet` files, and optionally creates diagnostic PNG plots.

2. **Step 2 : Raman matching**
   This script reads the processed `.parquet` spectra, compares unknown spectra with the reference library, calculates similarity scores, writes an Excel report, and optionally saves best-match PNG plots.

### Install all required libraries

Run this command before using the scripts:

```bash
pip install numpy pandas matplotlib scipy BaselineRemoval joblib tqdm pyarrow openpyxl xlsxwriter
```

### Libraries used

* `numpy` — numerical calculations and array operations.
* `pandas` — reading input tables, handling processed spectra, saving `.parquet` files, and creating Excel outputs.
* `matplotlib` — generating and saving diagnostic and best-match PNG plots.
* `scipy` — signal processing, smoothing, interpolation, sparse solvers, and similarity calculations.
* `BaselineRemoval` — baseline correction methods such as ZhangFit, ModPoly, and IModPoly.
* `joblib` — optional parallel processing to speed up batch preprocessing and plotting.
* `tqdm` — progress bars for preprocessing, matching, scoring, and plotting steps.
* `pyarrow` — required for reading and writing `.parquet` files with pandas.
* `openpyxl` — needed when reading `.xlsx` input files in Step 1.
* `xlsxwriter` — needed for writing the Step 2 Excel match report.

### Standard Python libraries

The scripts also use built-in Python libraries such as:

```text
os, re, sys, argparse, textwrap, contextlib
```

These are included with Python and do not need to be installed separately.



Step 1 : Raman preprocessing (baseline + smoothing + SNR)
====================================================================

In [17]:

from __future__ import annotations

import os
import argparse
import sys
from contextlib import contextmanager

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy import sparse
from scipy.sparse.linalg import spsolve
from BaselineRemoval import BaselineRemoval

try:
    from joblib import Parallel, delayed
    import joblib
    _HAVE_JOBLIB = True
except Exception:
    joblib = None
    _HAVE_JOBLIB = False

try:
    from tqdm.auto import tqdm
    _HAVE_TQDM = True
except Exception:
    tqdm = None
    _HAVE_TQDM = False


def _tqdm_is_on(cfg=None):
    if cfg is None:
        enabled = TQDM_ENABLED
    else:
        enabled = bool(cfg.get("tqdm_enabled", TQDM_ENABLED))
    return bool(enabled and _HAVE_TQDM)


def _iter_progress(iterable, desc, total=None, cfg=None):
    if not _tqdm_is_on(cfg):
        return iterable
    return tqdm(iterable, desc=desc, total=total, file=sys.stdout,
                dynamic_ncols=True, leave=True)


def _make_pbar(desc, total, cfg=None):
    if not _tqdm_is_on(cfg):
        return None
    return tqdm(desc=desc, total=total, file=sys.stdout,
                dynamic_ncols=True, leave=True)


@contextmanager
def tqdm_joblib(tqdm_object):
    if tqdm_object is None or joblib is None:
        yield tqdm_object
        return

    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

# ============================================================
# CONFIG  (edit these only)
# ============================================================

INPUT_DIR  = r"C:/Users/alqrifym-admin/Desktop/RamanAnalyzer_Fat/MicroPlastiX_Raman_only_Converted_raw/MicroPlastiX_Raman_only_Converted_raw"
OUTPUT_DIR = r"C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\PLOTS_5"

#Baseline method
# Full control (unlike the GUI, which is locked to ZhangFit):
#   'ZhangFit'  - adaptive iteratively-reweighted penalised least squares
#   'ModPoly'   - modified polynomial fit
#   'IModPoly'  - improved modified polynomial fit
#   'ALS'       - asymmetric least squares (sparse implementation below)
METHOD = "ZhangFit"

# ZhangFit params — engine default is lambda=50 (the GUI / engine value).
# (The Document-1 notebook used 100; set ZHANG_LAMBDA=100 to reproduce it.)
ZHANG_LAMBDA      = 50
ZHANG_PORDER      = 3
ZHANG_REPITITION  = 100

# ModPoly / IModPoly params
POLY_DEGREE = 3
ITERS       = 100
CONV_THRESH = 0.001

# ALS params
LAM_ALS = 10000
P_ALS   = 0.001

#Savitzky-Golay smoothing
# 'resolution_aware' - engine behaviour: physical width (cm^-1) - odd window
#                       from the median spacing (SG_TARGET_WIDTH_CM).
# 'fixed'            - Document-1 behaviour: a fixed odd window of SG_FIXED_WINDOW.
SG_MODE            = "resolution_aware"
SG_TARGET_WIDTH_CM = 18.0   # used in resolution_aware mode
SG_POLY_ORDER      = 3
SG_FIXED_WINDOW    = 21     # used in fixed mode
SG_FIXED_POLY      = 3      # used in fixed mode

#SNR windows
SNR_SIG_MIN   = 500.0
SNR_SIG_MAX   = 1800.0
SNR_NOISE_MIN = 1850.0
SNR_NOISE_MAX = 1950.0
SNR_THRESHOLD = 10.0

#SNR classification
SNR_MIN_SIGNAL_POINTS = 5
SNR_MIN_NOISE_POINTS  = 5
SNR_NOISE_FLOOR_VALUE = 1e-4   # on the [0, 1] normalised scale
SNR_CAP               = 1000.0

#Output toggles
SAVE_PLOTS         = True
PLOTS_ONLY_LOW_SNR = False
TQDM_ENABLED       = True      # True - show proper tqdm progress bars for each step
N_WORKERS          = None      # None - os.cpu_count()

#Plot style
PLOT_DISPLAY_MIN = None        # x-axis min for plots; None - spectrum minimum
PLOT_DISPLAY_MAX = None        # x-axis max for plots; None - spectrum maximum

SUPPORTED_EXTS = {".csv", ".xlsx", ".txt"}


# PLOT STYLE
from matplotlib import rcParams

rcParams["font.family"]       = "serif"
rcParams["font.serif"]        = ["Times New Roman", "DejaVu Serif"]
rcParams["font.size"]         = 16
rcParams["axes.linewidth"]    = 1.1
rcParams["axes.labelsize"]    = 18
rcParams["axes.titlesize"]    = 18
rcParams["xtick.labelsize"]   = 15
rcParams["ytick.labelsize"]   = 15
rcParams["xtick.direction"]   = "in"
rcParams["ytick.direction"]   = "in"
rcParams["xtick.major.size"]  = 6
rcParams["ytick.major.size"]  = 6
rcParams["xtick.major.width"] = 1.1
rcParams["ytick.major.width"] = 1.1
rcParams["legend.frameon"]    = False
rcParams["legend.fontsize"]   = 14

COLOR_RAW       = "#7a7a7a"
COLOR_BASELINE  = "#3a8f5a"
COLOR_CORRECTED = "#1f4f8b"
COLOR_SMOOTHED  = "#1f3f72"
COLOR_SIGNAL    = "#9ec5e8"
COLOR_NOISE     = "#b03030"
COLOR_PEAK      = "#1f3f72"



# BASELINE METHODS
def baseline_als(y, lam, p, niter=10):
    """Asymmetric Least Squares baseline (sparse, no dense LxL allocation).
    Ported from the Document-1 notebook."""
    y = np.asarray(y, dtype=float)
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L), format="csc")
    DtD = lam * (D.T @ D)
    w = np.ones(L)
    z = y
    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        Z = W + DtD
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z


def compute_baseline_corrected(intensity, method, cfg):
    """Return the baseline-corrected spectrum for the chosen method.
    'baseline' is recovered as intensity - corrected downstream."""
    if method == "ALS":
        baseline = baseline_als(intensity, cfg["lam_als"], cfg["p_als"])
        return intensity - baseline

    baseObj = BaselineRemoval(intensity)
    if method == "ModPoly":
        return baseObj.ModPoly(cfg["poly_degree"], cfg["iters"], cfg["conv_thresh"])
    if method == "IModPoly":
        return baseObj.IModPoly(cfg["poly_degree"], cfg["iters"], cfg["conv_thresh"])
    if method == "ZhangFit":
        return baseObj.ZhangFit(lambda_=cfg["zhang_lambda"],
                                porder=cfg["zhang_porder"],
                                repitition=cfg["zhang_repitition"])
    raise ValueError(f"Unknown baseline method: {method!r}")


def _read_raw_table(file_path: str):
    """Delimited read with no header/type assumptions. Returns str DataFrame."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".xlsx":
        return pd.read_excel(file_path, header=None, dtype=str)

    import re
    from io import StringIO

    with open(file_path, "r", encoding="utf-8", errors="replace") as fh:
        lines = [ln for ln in fh.readlines() if not ln.lstrip().startswith("#")]
    if not lines:
        raise ValueError(f"Empty file: {file_path}")
    text_buf = "".join(lines)

    candidates = []
    for sep in (",", "\t", ";", r"\s+"):
        if sep == r"\s+":
            splitter = lambda ln: re.split(r"\s+", ln.strip()) if ln.strip() else []
        else:
            splitter = lambda ln: ln.rstrip("\n").rstrip("\r").split(sep)
        max_cols = 0
        for ln in lines:
            w = len(splitter(ln))
            if w > max_cols:
                max_cols = w
        if max_cols < 2:
            continue
        try:
            df = pd.read_csv(
                StringIO(text_buf), sep=sep, header=None,
                names=list(range(max_cols)), dtype=str,
                engine="python" if sep == r"\s+" else "c",
                skip_blank_lines=False, on_bad_lines="skip",
            )
        except Exception:
            continue
        if df.shape[0] == 0 or df.shape[1] < 2:
            continue

        numeric_rows = 0
        for i in range(min(len(df), 200)):
            n = 0
            for v in df.iloc[i].values:
                if v is None:
                    continue
                s = str(v).strip()
                if not s:
                    continue
                try:
                    if np.isfinite(float(s)):
                        n += 1
                except (ValueError, TypeError):
                    pass
                if n >= 2:
                    break
            if n >= 2:
                numeric_rows += 1
        candidates.append((numeric_rows, df.shape[1], df))

    if not candidates:
        raise ValueError(f"Could not parse: {file_path}")
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2]


def _looks_like_header_row(values):
    keywords = ("raman", "shift", "intensity", "wave", "wavenumber",
                "cm-1", "1/cm", "counts", "cps")
    for v in values:
        if v is None:
            continue
        s = str(v).strip().lower()
        if s and any(k in s for k in keywords):
            return True
    return False


def _row_numeric_count(values):
    n = 0
    for v in values:
        if v is None:
            continue
        s = str(v).strip()
        if not s:
            continue
        try:
            if np.isfinite(float(s)):
                n += 1
        except (ValueError, TypeError):
            pass
    return n


def load_data(file_path: str):
    """Robust loader for messy Raman exports. Returns ascending, deduped
    DataFrame with columns ['Wave', 'Intensity']."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext not in SUPPORTED_EXTS:
        raise ValueError(f"Unsupported format: {ext}")

    # Fast path: a previously-exported intermediate with Wave + Smoothed.
    try:
        quick = (pd.read_excel(file_path, nrows=5) if ext == ".xlsx"
                 else pd.read_csv(file_path, nrows=5))
        if "Wave" in quick.columns and "Smoothed" in quick.columns:
            full = (pd.read_excel(file_path) if ext == ".xlsx"
                    else pd.read_csv(file_path))
            out = full[["Wave", "Smoothed"]].copy()
            out.columns = ["Wave", "Intensity"]
            out["Wave"] = pd.to_numeric(out["Wave"], errors="coerce")
            out["Intensity"] = pd.to_numeric(out["Intensity"], errors="coerce")
            out = out.dropna()
            if len(out) == 0:
                raise ValueError("No valid numeric data")
            out = out.sort_values("Wave", kind="mergesort").reset_index(drop=True)
            return out.drop_duplicates(subset="Wave", keep="first").reset_index(drop=True)
    except Exception:
        pass

    raw = _read_raw_table(file_path)

    def _col_is_empty(col):
        for v in col:
            if v is None:
                continue
            if isinstance(v, float) and np.isnan(v):
                continue
            if str(v).strip() == "":
                continue
            return False
        return True

    keep_cols = [c for c in raw.columns if not _col_is_empty(raw[c])]
    if len(keep_cols) < 2:
        raise ValueError(f"Fewer than 2 non-empty columns in: {file_path}")
    raw = raw[keep_cols].reset_index(drop=True)

    data_start = None
    for i in range(len(raw)):
        if _row_numeric_count(raw.iloc[i].values) >= 2:
            data_start = i
            break
    if data_start is None:
        raise ValueError(f"No numeric data rows found in: {file_path}")

    header_row = None
    if data_start > 0:
        cand = raw.iloc[data_start - 1].values
        if _looks_like_header_row(cand):
            header_row = cand

    data_block = raw.iloc[data_start:].reset_index(drop=True)
    numeric = data_block.apply(lambda c: pd.to_numeric(c, errors="coerce"))
    finite_counts = numeric.notna().sum()
    valid_cols = [c for c in numeric.columns if finite_counts[c] > 0]
    if len(valid_cols) < 2:
        raise ValueError(f"Fewer than 2 numeric columns in: {file_path}")
    numeric = numeric[valid_cols]

    ranked = sorted(valid_cols,
                    key=lambda c: (-int(finite_counts[c]), valid_cols.index(c)))
    wave_col, int_col = ranked[0], ranked[1]

    if header_row is not None:
        header_map = {}
        for pos, col_name in enumerate(raw.columns):
            if pos < len(header_row):
                header_map[col_name] = str(header_row[pos]).strip().lower()
        wave_kw = ("raman", "shift", "wave", "cm-1", "1/cm", "wavenumber")
        int_kw = ("intensity", "counts", "cps")
        wave_cand = int_cand = None
        for c in valid_cols:
            h = header_map.get(c, "")
            if wave_cand is None and any(k in h for k in wave_kw):
                wave_cand = c
            if int_cand is None and any(k in h for k in int_kw):
                int_cand = c
        if wave_cand is not None and int_cand is not None and wave_cand != int_cand:
            wave_col, int_col = wave_cand, int_cand

    out = pd.DataFrame({
        "Wave": numeric[wave_col].values,
        "Intensity": numeric[int_col].values,
    }).dropna(subset=["Wave", "Intensity"]).reset_index(drop=True)
    if len(out) == 0:
        raise ValueError(f"No numeric data rows found in: {file_path}")

    out = out.sort_values("Wave", kind="mergesort").reset_index(drop=True)
    out = out.drop_duplicates(subset="Wave", keep="first").reset_index(drop=True)
    return out



# RESOLUTION-AWARE SAVITZKY-GOLAY
def sg_window_length(wavenumbers, target_width_cm, poly_order):
    """Physical smoothing width (cm^-1) - valid odd SG window from median spacing."""
    wn = np.asarray(wavenumbers, dtype=float)
    if len(wn) < 5:
        n = len(wn)
        if n < 3:
            return n
        return n if n % 2 == 1 else n - 1

    diffs = np.diff(wn)
    diffs = diffs[np.isfinite(diffs)]
    diffs = diffs[np.abs(diffs) > 0]
    if len(diffs) == 0:
        wl = poly_order + 2
        return wl + 1 if wl % 2 == 0 else wl

    dx = float(np.median(np.abs(diffs)))
    wl = int(round(target_width_cm / dx))
    if wl % 2 == 0:
        wl += 1

    min_valid = poly_order + 2
    if min_valid % 2 == 0:
        min_valid += 1
    wl = max(wl, min_valid)

    max_valid = len(wn) if len(wn) % 2 == 1 else len(wn) - 1
    wl = min(wl, max_valid)

    if wl <= poly_order:
        wl = poly_order + 2
        if wl % 2 == 0:
            wl += 1
        if wl > max_valid:
            wl = max_valid
    if wl < 3 and len(wn) >= 3:
        wl = 3
    return int(wl)


def sg_smooth_resolution_aware(y, wavenumbers, target_width_cm, poly_order):
    """Returns (smoothed, window_length_used). On failure: (copy_of_y, nan)."""
    y = np.asarray(y, dtype=float)
    wn = np.asarray(wavenumbers, dtype=float)
    n = len(y)
    if n < 5:
        return y.copy(), float("nan")
    wl = sg_window_length(wn, target_width_cm, poly_order)
    po = int(poly_order)
    if wl is None or wl <= po or wl < 3 or wl > n:
        return y.copy(), float("nan")
    try:
        return savgol_filter(y, wl, po), float(wl)
    except Exception:
        return y.copy(), float("nan")


def sanitize_fixed_window(window_length, poly_order, n_points):
    """Coerce (window, poly) into something savgol_filter accepts. Returns the
    sanitised odd window length, or None if no valid config exists."""
    if n_points < 3:
        return None
    wl = int(window_length)
    po = int(poly_order)
    if wl % 2 == 0:
        wl += 1
    min_valid = po + 2
    if min_valid % 2 == 0:
        min_valid += 1
    if wl < min_valid:
        wl = min_valid
    max_valid = n_points if n_points % 2 == 1 else n_points - 1
    if wl > max_valid:
        wl = max_valid
    if wl < 3 or wl <= po:
        return None
    return wl


def sg_smooth(y, wavenumbers, cfg):
    """Dispatch SG smoothing on cfg['sg_mode']. Returns (smoothed, window_used)."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 5:
        return y.copy(), float("nan")

    if cfg["sg_mode"] == "fixed":
        wl = sanitize_fixed_window(cfg["sg_fixed_window"], cfg["sg_fixed_poly"], n)
        po = int(cfg["sg_fixed_poly"])
        if wl is None:
            return y.copy(), float("nan")
        try:
            return savgol_filter(y, wl, po), float(wl)
        except Exception:
            return y.copy(), float("nan")

    # default: resolution_aware
    return sg_smooth_resolution_aware(
        y, wavenumbers, cfg["sg_target_width_cm"], cfg["sg_poly_order"])



# RANGE-AWARE SNR 
def inline_snr(wavenumbers, corrected, smoothed,
               sig_min, sig_max, noise_min, noise_max):
    wn = np.asarray(wavenumbers, dtype=float)
    sm = np.asarray(smoothed, dtype=float)
    co = np.asarray(corrected, dtype=float)

    sig_mask = (wn >= sig_min) & (wn <= sig_max)
    n_mask = (wn >= noise_min) & (wn <= noise_max)
    if (int(sig_mask.sum()) < SNR_MIN_SIGNAL_POINTS
            or int(n_mask.sum()) < SNR_MIN_NOISE_POINTS):
        return float("nan")

    s_min = float(np.nanmin(sm))
    s_max = float(np.nanmax(sm))
    rng = s_max - s_min
    if not np.isfinite(rng) or rng <= 0:
        return float("nan")

    if abs(s_min) < 1e-6 and abs(s_max - 1.0) < 1e-6:
        sm_n = sm.copy()
        co_n = co.copy()
    else:
        sm_n = (sm - s_min) / rng
        co_n = (co - s_min) / rng

    signal_peak = float(np.max(sm_n[sig_mask]))
    region = co_n[n_mask]
    noise_val = float(np.sqrt(np.mean((region - region.mean()) ** 2)))
    if not np.isfinite(noise_val):
        return float("nan")

    if noise_val < SNR_NOISE_FLOOR_VALUE:
        snr = signal_peak / SNR_NOISE_FLOOR_VALUE
        return float(min(snr, SNR_CAP))
    return float(signal_peak / noise_val)


def discover_input_files(input_root):
    records = []
    for root, _, files in os.walk(input_root):
        for f in files:
            if os.path.splitext(f)[1].lower() not in SUPPORTED_EXTS:
                continue
            rel_dir = os.path.relpath(root, input_root)
            rel_dir = "" if rel_dir == "." else rel_dir
            records.append({"input_path": os.path.join(root, f),
                            "rel_dir": rel_dir, "file_name": f})
    return sorted(records, key=lambda x: (x["rel_dir"], x["file_name"]))



# Files Processing
def process_one(rec, output_root, cfg):
    file_path = rec["input_path"]
    rel_dir = rec["rel_dir"]
    try:
        data = load_data(file_path)
        intensity = data["Intensity"].values.astype(float)
        wavenumbers = data["Wave"].values.astype(float)

        corrected = compute_baseline_corrected(intensity, cfg["method"], cfg)
        baseline = intensity - corrected

        smoothed, _wl = sg_smooth(corrected, wavenumbers, cfg)

        snr_val = inline_snr(
            wavenumbers, corrected, smoothed,
            cfg["sig_min"], cfg["sig_max"],
            cfg["noise_min"], cfg["noise_max"],
        )
        snr_finite = bool(np.isfinite(snr_val))
        snr_flag = ("UNMEASURABLE" if not snr_finite
                    else ("LOW" if snr_val < cfg["snr_threshold"] else "OK"))

        out_dir = os.path.join(output_root, rel_dir) if rel_dir else output_root
        os.makedirs(out_dir, exist_ok=True)
        base = os.path.splitext(os.path.basename(file_path))[0]
        parquet_path = os.path.join(out_dir, f"{base}_bs+sm.parquet")

        processed = pd.DataFrame({
            "Wave": wavenumbers,
            "Original_Intensity": intensity,
            "Baseline": baseline,
            "Baseline_Corrected": corrected,
            "Smoothed": smoothed,
            "SNR": (round(float(snr_val), 2) if snr_finite else np.nan),
            "SNR_Flag": snr_flag,
            "SNR_Threshold": float(cfg["snr_threshold"]),
        })
        processed.to_parquet(parquet_path, index=False)

        return (True, {
            "File": os.path.basename(file_path),
            "Relative_Folder": rel_dir if rel_dir else "[ROOT]",
            "Output_Parquet": parquet_path,
            "SNR": snr_val,
            "SNR_Flag": snr_flag,
            "Low_SNR_Flag": bool(snr_flag == "LOW"),
        })
    except Exception as e:
        return (False, {
            "File": os.path.basename(file_path),
            "Relative_Folder": rel_dir if rel_dir else "[ROOT]",
            "Error": str(e),
        })



# Diagnostic plots
def _snr_components(wavenumbers, corrected, smoothed, cfg):
    """Recompute normalised signal-peak and noise from stored arrays so the
    SNR panel can show the breakdown. Mirrors inline_snr's normalisation."""
    wn = np.asarray(wavenumbers, dtype=float)
    sm = np.asarray(smoothed, dtype=float)
    co = np.asarray(corrected, dtype=float)
    sig_mask = (wn >= cfg["sig_min"]) & (wn <= cfg["sig_max"])
    n_mask = (wn >= cfg["noise_min"]) & (wn <= cfg["noise_max"])
    if int(sig_mask.sum()) < 1 or int(n_mask.sum()) < 1:
        return np.nan, np.nan
    s_min, s_max = float(np.nanmin(sm)), float(np.nanmax(sm))
    rng = s_max - s_min
    if not np.isfinite(rng) or rng <= 0:
        return np.nan, np.nan
    if abs(s_min) < 1e-6 and abs(s_max - 1.0) < 1e-6:
        sm_n, co_n = sm.copy(), co.copy()
    else:
        sm_n = (sm - s_min) / rng
        co_n = (co - s_min) / rng
    signal_peak = float(np.max(sm_n[sig_mask]))
    region = co_n[n_mask]
    noise_val = float(np.sqrt(np.mean((region - region.mean()) ** 2)))
    return signal_peak, noise_val


def _save_fig(fig, out_base, cfg):
    """Save PNG only. PDF export has been removed completely."""
    fig.savefig(out_base + ".png", dpi=300, bbox_inches="tight")
    plt.close(fig)


def _plot_preprocessing(wn, intensity, baseline, corrected, smoothed,
                        stem, method, dmin, dmax, out_base, cfg):
    fig, (ax_top, ax_bot) = plt.subplots(
        nrows=2, ncols=1, figsize=(13.5, 12.5),
        gridspec_kw={"height_ratios": [1.0, 1.0], "hspace": 0.50})
    fig.subplots_adjust(left=0.085, right=0.97, top=0.92, bottom=0.07)

    ax_top.plot(wn, intensity, color=COLOR_RAW, lw=1.0, alpha=0.9,
                label="Raw spectrum")
    ax_top.plot(wn, baseline, color=COLOR_BASELINE, lw=1.6, ls="--",
                label="Baseline")
    ax_top.plot(wn, corrected, color=COLOR_CORRECTED, lw=1.2, alpha=0.95,
                label="Raw spectrum \u2212 baseline")
    ax_top.set_title(f"Spectrum analysis using {method} method",
                     pad=58, fontsize=18, fontweight="bold")
    ax_top.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=18, labelpad=10)
    ax_top.set_ylabel("Intensity", fontsize=18, labelpad=10)
    ax_top.set_xlim(dmin, dmax)
    ax_top.grid(True, linestyle=":", linewidth=0.6, color="#bbbbbb", alpha=0.6)
    ax_top.legend(loc="lower center", bbox_to_anchor=(0.5, 1.005), ncol=3,
                  fontsize=13, framealpha=0.0, handletextpad=0.6,
                  columnspacing=1.5, borderaxespad=0.0)
    for s in ("top", "right"):
        ax_top.spines[s].set_visible(False)

    ax_bot.plot(wn, smoothed, color=COLOR_SMOOTHED, lw=1.6, ls="--",
                label="Smoothed data")
    ax_bot.set_title("Smoothing using Savitzky\u2013Golay filter",
                     pad=58, fontsize=18, fontweight="bold")
    ax_bot.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=18, labelpad=10)
    ax_bot.set_ylabel("Intensity", fontsize=18, labelpad=10)
    ax_bot.set_xlim(dmin, dmax)
    ax_bot.grid(True, linestyle=":", linewidth=0.6, color="#bbbbbb", alpha=0.6)
    ax_bot.legend(loc="lower center", bbox_to_anchor=(0.5, 1.005), ncol=1,
                  fontsize=13, framealpha=0.0, handletextpad=0.6,
                  borderaxespad=0.0)
    for s in ("top", "right"):
        ax_bot.spines[s].set_visible(False)

    _save_fig(fig, out_base, cfg)


def _plot_snr(wn, corrected, smoothed, snr, stem, dmin, dmax, out_base, cfg):
    sig_peak_n, noise_n = _snr_components(wn, corrected, smoothed, cfg)
    thr = cfg["snr_threshold"]
    if not np.isfinite(snr):
        verdict, vcolor, snr_text = "UNMEASURABLE", "#666666", "SNR = NaN  (unmeasurable)"
    elif snr < thr:
        verdict, vcolor, snr_text = f"FAIL  (SNR < {thr:g})", "#b03030", f"SNR = {snr:.1f}"
    else:
        verdict, vcolor, snr_text = f"PASS  (SNR \u2265 {thr:g})", "#1b7a3e", f"SNR = {snr:.1f}"

    fig, ax = plt.subplots(figsize=(13.5, 7.5))
    fig.subplots_adjust(left=0.085, right=0.97, top=0.80, bottom=0.13)

    s_lo, s_hi = cfg["sig_min"], cfg["sig_max"]
    n_lo, n_hi = cfg["noise_min"], cfg["noise_max"]
    ax.axvspan(s_lo, s_hi, alpha=0.30, color=COLOR_SIGNAL,
               label=f"Signal window [{s_lo:.0f}\u2013{s_hi:.0f}] cm$^{{-1}}$", zorder=1)
    ax.axvspan(n_lo, n_hi, alpha=0.22, color=COLOR_NOISE,
               label=f"Noise window [{n_lo:.0f}\u2013{n_hi:.0f}] cm$^{{-1}}$  (flat_region)",
               zorder=1)
    ax.plot(wn, smoothed, color=COLOR_SMOOTHED, lw=1.5,
            label="Smoothed spectrum", zorder=3)

    sig_mask = (wn >= s_lo) & (wn <= s_hi)
    if int(sig_mask.sum()) > 0:
        idx = int(np.argmax(smoothed[sig_mask]))
        peak_w = float(wn[sig_mask][idx]); peak_y = float(smoothed[sig_mask][idx])
        ax.axvline(peak_w, color=COLOR_PEAK, lw=1.0, ls="--", alpha=0.65, zorder=2,
                   label=f"Peak @ {peak_w:.0f} cm$^{{-1}}$")
        ax.scatter([peak_w], [peak_y], color=COLOR_PEAK, s=70, zorder=4,
                   edgecolor="white", lw=0.9)

    ax.set_title(f"SNR diagnostic  |  {snr_text}  \u2014  {verdict}",
                 pad=58, fontsize=18, fontweight="bold", color=vcolor)
    ax.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=18, labelpad=10)
    ax.set_ylabel("Intensity", fontsize=18, labelpad=10)
    # Always keep the noise window visible even if it sits outside the display.
    xlo = min(dmin, s_lo, n_lo); xhi = max(dmax, s_hi, n_hi)
    pad = 0.02 * (xhi - xlo)
    ax.set_xlim(xlo - pad, xhi + pad)
    ax.grid(True, linestyle=":", linewidth=0.6, color="#bbbbbb", alpha=0.6)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.005), ncol=4,
              fontsize=13, framealpha=0.0, handletextpad=0.6,
              columnspacing=1.5, borderaxespad=0.0)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

    if np.isfinite(snr):
        sig_str = f"{sig_peak_n:.3f}" if np.isfinite(sig_peak_n) else "NaN"
        noise_str = f"{noise_n:.4f}" if np.isfinite(noise_n) else "NaN"
        info = (f"signal peak (normalised) = {sig_str}\n"
                f"noise (flat_region, normalised) = {noise_str}\n"
                f"SNR = signal / noise = {snr:.1f}")
        ax.text(0.985, 0.04, info, transform=ax.transAxes, ha="right", va="bottom",
                fontsize=12, family="monospace", color="#222222",
                bbox=dict(facecolor="white", edgecolor="#888888",
                          linewidth=0.6, pad=6.0, alpha=0.92), zorder=5)

    _save_fig(fig, out_base, cfg)


def plot_one(row, cfg):
    try:
        df = pd.read_parquet(row["Output_Parquet"])
        wn = df["Wave"].values
        intensity = df["Original_Intensity"].values
        baseline = df["Baseline"].values
        corrected = df["Baseline_Corrected"].values
        smoothed = df["Smoothed"].values
        snr = row["SNR"]

        stem = os.path.splitext(os.path.basename(row["Output_Parquet"]))[0]
        if stem.endswith("_bs+sm"):
            stem = stem[:-6]

        # Single consolidated plots root, mirroring any input subfolders to
        # avoid filename collisions.
        plot_dir = os.path.join(cfg["output_dir"], "plots")
        rel = row["Relative_Folder"]
        if rel and rel != "[ROOT]":
            plot_dir = os.path.join(plot_dir, rel)
        os.makedirs(plot_dir, exist_ok=True)

        dmin = cfg["plot_display_min"] if cfg["plot_display_min"] is not None else float(np.min(wn))
        dmax = cfg["plot_display_max"] if cfg["plot_display_max"] is not None else float(np.max(wn))

        _plot_preprocessing(wn, intensity, baseline, corrected, smoothed,
                            stem, cfg["method"], dmin, dmax,
                            os.path.join(plot_dir, f"{stem}_preprocessing"), cfg)
        _plot_snr(wn, corrected, smoothed, snr, stem, dmin, dmax,
                  os.path.join(plot_dir, f"{stem}_snr"), cfg)

        return (True, row["Output_Parquet"])
    except Exception as e:
        return (False, {"Parquet": row["Output_Parquet"], "Error": str(e)})


def run(cfg):
    input_dir = cfg["input_dir"]
    output_dir = cfg["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    print("=" * 60)
    print("SCRIPT VERSION: STEP1_PNG_ONLY_WITH_TQDM")
    print(f"PLOTTING: {'ON' if cfg['save_plots'] else 'OFF'}")
    if cfg["save_plots"]:
        print("Plot format        - PNG only")
        print(f"Plot root          - {os.path.join(output_dir, 'plots')}")
        print(f"Plots only low SNR - {'ON' if cfg['plots_only_low_snr'] else 'OFF'}")
    else:
        print("Plot output        - disabled")
    print(f"TQDM PROGRESS: {'ON' if _tqdm_is_on(cfg) else 'OFF'}")
    if cfg.get("tqdm_enabled", True) and not _HAVE_TQDM:
        print("TQDM note          - tqdm is not installed; install with: pip install tqdm")
    print("=" * 60)

    files = discover_input_files(input_dir)
    if not files:
        print(f"No supported files found in {input_dir}")
        return

    n_workers = cfg["n_workers"] or (os.cpu_count() or 4)
    total = len(files)
    print(f"Found {total} files | workers: {n_workers}")
    if cfg["method"] == "ZhangFit":
        print(f"Baseline: ZhangFit lambda={cfg['zhang_lambda']}, "
              f"porder={cfg['zhang_porder']}, reps={cfg['zhang_repitition']}")
    elif cfg["method"] in ("ModPoly", "IModPoly"):
        print(f"Baseline: {cfg['method']} degree={cfg['poly_degree']}, "
              f"iters={cfg['iters']}, conv={cfg['conv_thresh']}")
    elif cfg["method"] == "ALS":
        print(f"Baseline: ALS lam={cfg['lam_als']}, p={cfg['p_als']}")
    if cfg["sg_mode"] == "fixed":
        print(f"SG: fixed window {cfg['sg_fixed_window']}, order {cfg['sg_fixed_poly']}")
    else:
        print(f"SG: resolution-aware {cfg['sg_target_width_cm']} cm^-1, "
              f"order {cfg['sg_poly_order']}")
    print(f"Signal {cfg['sig_min']:.0f}-{cfg['sig_max']:.0f} | "
          f"Noise {cfg['noise_min']:.0f}-{cfg['noise_max']:.0f} | "
          f"threshold {cfg['snr_threshold']}")
    print("=" * 60)

    # Phase 1: process
    print("[Phase 1] Baseline + smoothing + SNR ...")
    if _HAVE_JOBLIB and n_workers != 1:
        with tqdm_joblib(_make_pbar("Phase 1 preprocessing", total, cfg)):
            results = Parallel(n_jobs=n_workers, backend="loky")(
                delayed(process_one)(rec, output_dir, cfg) for rec in files)
    else:
        results = [
            process_one(rec, output_dir, cfg)
            for rec in _iter_progress(files, "Phase 1 preprocessing", total, cfg)
        ]

    ok_rows = []
    skipped = []
    for ok, info in _iter_progress(results, "Collect process results", len(results), cfg):
        if ok:
            ok_rows.append(info)
        else:
            skipped.append(info)

    #SNR summary
    if ok_rows:
        summary_records = []
        for r in _iter_progress(ok_rows, "Build SNR summary table", len(ok_rows), cfg):
            summary_records.append({
                "File": r["File"],
                "Relative_Folder": r["Relative_Folder"],
                "SNR": (round(r["SNR"], 2) if np.isfinite(r["SNR"]) else np.nan),
                "SNR_Flag": r["SNR_Flag"],
                "Output_Parquet": r["Output_Parquet"],
            })

        snr_df = pd.DataFrame(summary_records).sort_values(
            "SNR", ascending=True, na_position="first")
        snr_csv = os.path.join(output_dir, "snr_summary.csv")
        snr_df.to_csv(snr_csv, index=False)
        n_low = int((snr_df["SNR_Flag"] == "LOW").sum())
        n_unmeas = int((snr_df["SNR_Flag"] == "UNMEASURABLE").sum())

        print("[Summary plot] SNR distribution ...")
        fig, ax = plt.subplots(figsize=(12, 5))
        colors = [
            "tomato" if f == "LOW" else ("lightgrey" if f == "UNMEASURABLE" else "steelblue")
            for f in _iter_progress(snr_df["SNR_Flag"], "Build SNR plot colors", len(snr_df), cfg)
        ]
        ax.bar(range(len(snr_df)), snr_df["SNR"].fillna(0), color=colors)
        ax.axhline(cfg["snr_threshold"], color="red", ls="--", lw=1.5,
                   label=f"Threshold = {cfg['snr_threshold']}")
        ax.set_xlabel("File index (sorted by SNR)")
        ax.set_ylabel("SNR")
        ax.set_title(f"SNR distribution -- {n_low} low, {n_unmeas} unmeasurable")
        ax.legend()
        fig.tight_layout()
        plots_root = os.path.join(output_dir, "plots")
        os.makedirs(plots_root, exist_ok=True)
        fig.savefig(os.path.join(plots_root, "snr_distribution.png"), dpi=120)
        plt.close(fig)
        print(f"SNR summary - {snr_csv}  ({n_low} low, {n_unmeas} unmeasurable)")

    #plots
    if cfg["save_plots"] and ok_rows:
        to_plot = [
            r for r in _iter_progress(ok_rows, "Select rows to plot", len(ok_rows), cfg)
            if (not cfg["plots_only_low_snr"]) or r["Low_SNR_Flag"]
        ]
        print(f"[Phase 2] Plotting {len(to_plot)} diagnostics ...")
        if _HAVE_JOBLIB and n_workers != 1:
            with tqdm_joblib(_make_pbar("Phase 2 diagnostic PNGs", len(to_plot), cfg)):
                plot_res = Parallel(n_jobs=n_workers, backend="loky")(
                    delayed(plot_one)(r, cfg) for r in to_plot)
        else:
            plot_res = [
                plot_one(r, cfg)
                for r in _iter_progress(to_plot, "Phase 2 diagnostic PNGs", len(to_plot), cfg)
            ]

        plot_fail = []
        for ok, info in _iter_progress(plot_res, "Collect plot results", len(plot_res), cfg):
            if not ok:
                plot_fail.append(info)
        for p in plot_fail:
            print(f"  plot error: {p['Parquet']} - {p['Error']}")

    #Summary
    print("=" * 60)
    print(f"Done: {len(ok_rows)}/{total} processed")
    if skipped:
        print(f"Skipped ({len(skipped)}):")
        for s in skipped:
            print(f"  x {s['Relative_Folder']} | {s['File']} - {s['Error']}")
    print(f"Output - {output_dir}")

def build_cfg(args):
    return {
        "input_dir": args.input,
        "output_dir": args.output,
        "method": args.method,
        "zhang_lambda": args.zhang_lambda,
        "zhang_porder": ZHANG_PORDER,
        "zhang_repitition": ZHANG_REPITITION,
        "poly_degree": args.poly_degree,
        "iters": args.iters,
        "conv_thresh": args.conv_thresh,
        "lam_als": args.lam_als,
        "p_als": args.p_als,
        "sg_mode": args.sg_mode,
        "sg_target_width_cm": args.sg_width,
        "sg_poly_order": SG_POLY_ORDER,
        "sg_fixed_window": args.sg_fixed_window,
        "sg_fixed_poly": args.sg_fixed_poly,
        "sig_min": args.sig_min, "sig_max": args.sig_max,
        "noise_min": args.noise_min, "noise_max": args.noise_max,
        "snr_threshold": args.threshold,
        "save_plots": not args.no_plots,
        "plots_only_low_snr": args.plots_only_low_snr,
        "tqdm_enabled": not args.no_tqdm,
        "plot_display_min": args.display_min,
        "plot_display_max": args.display_max,
        "n_workers": args.workers,
    }


def main():
    ap = argparse.ArgumentParser(description="Step 1 Raman preprocessing (headless).")
    ap.add_argument("--input", default=INPUT_DIR)
    ap.add_argument("--output", default=OUTPUT_DIR)
    ap.add_argument("--method", default=METHOD,
                    choices=["ZhangFit", "ModPoly", "IModPoly", "ALS"])
    ap.add_argument("--zhang-lambda", type=float, default=ZHANG_LAMBDA, dest="zhang_lambda")
    ap.add_argument("--poly-degree", type=int, default=POLY_DEGREE, dest="poly_degree")
    ap.add_argument("--iters", type=int, default=ITERS)
    ap.add_argument("--conv-thresh", type=float, default=CONV_THRESH, dest="conv_thresh")
    ap.add_argument("--lam-als", type=float, default=LAM_ALS, dest="lam_als")
    ap.add_argument("--p-als", type=float, default=P_ALS, dest="p_als")
    ap.add_argument("--sg-mode", default=SG_MODE,
                    choices=["resolution_aware", "fixed"], dest="sg_mode")
    ap.add_argument("--sg-width", type=float, default=SG_TARGET_WIDTH_CM, dest="sg_width")
    ap.add_argument("--sg-fixed-window", type=int, default=SG_FIXED_WINDOW, dest="sg_fixed_window")
    ap.add_argument("--sg-fixed-poly", type=int, default=SG_FIXED_POLY, dest="sg_fixed_poly")
    ap.add_argument("--sig-min", type=float, default=SNR_SIG_MIN, dest="sig_min")
    ap.add_argument("--sig-max", type=float, default=SNR_SIG_MAX, dest="sig_max")
    ap.add_argument("--noise-min", type=float, default=SNR_NOISE_MIN, dest="noise_min")
    ap.add_argument("--noise-max", type=float, default=SNR_NOISE_MAX, dest="noise_max")
    ap.add_argument("--threshold", type=float, default=SNR_THRESHOLD)
    ap.add_argument("--workers", type=int, default=N_WORKERS)
    ap.add_argument("--no-plots", action="store_true", default=not SAVE_PLOTS)
    ap.add_argument("--plots-only-low-snr", action="store_true",
                    default=PLOTS_ONLY_LOW_SNR, dest="plots_only_low_snr")
    ap.add_argument("--no-tqdm", action="store_true", default=not TQDM_ENABLED,
                    help="Disable tqdm progress bars.")
    ap.add_argument("--display-min", type=float, default=PLOT_DISPLAY_MIN, dest="display_min")
    ap.add_argument("--display-max", type=float, default=PLOT_DISPLAY_MAX, dest="display_max")
    args, _ = ap.parse_known_args()
    run(build_cfg(args))


if __name__ == "__main__":
    main()

SCRIPT VERSION: STEP1_PNG_ONLY_WITH_TQDM
PLOTTING: ON
Plot format        - PNG only
Plot root          - C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\PLOTS_5\plots
Plots only low SNR - OFF
TQDM PROGRESS: ON
Found 39 files | workers: 12
Baseline: ZhangFit lambda=50, porder=3, reps=100
SG: resolution-aware 18.0 cm^-1, order 3
Signal 500-1800 | Noise 1850-1950 | threshold 10.0
[Phase 1] Baseline + smoothing + SNR ...
Build SNR summary table: 100%|██████████████████████████████████████████████████████| 39/39 [00:00<00:00, 39077.37it/s]
[Summary plot] SNR distribution ...
Build SNR plot colors: 100%|████████████████████████████████████████████████████████| 39/39 [00:00<00:00, 38762.53it/s]
SNR summary - C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\PLOTS_5\snr_summary.csv  (0 low, 10 unmeasurable)
Select rows to plot: 100%|█████████████████████████████████████████████████████████████████████| 39/39 [00:00<?, ?it/s]
[Phase 2] Plotting 39 diagnostics

## Step 2 : Spectra Matching 

In [15]:
from __future__ import annotations

import os
import re
import sys
import argparse
import textwrap

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.stats import pearsonr

try:
    from tqdm import tqdm
    _TQDM_AVAILABLE = True
except ImportError:
    tqdm = None
    _TQDM_AVAILABLE = False


#UNKNOWN_FOLDER = r"UNKNOWN_PARQUET_FOLDER_HERE"
UNKNOWN_FOLDER = r"C:/Users/alqrifym-admin/Desktop/RamanAnalyzer_Fat/MicroPlastiX_Raman_only_Converted_raw/MicroPlastiX_Raman_only_Converted_raw/unknown_baseline_smoothed"
#KNOWN_ROOT     = r"REFERENCE_LIBRARY_ROOT_HERE"
KNOWN_ROOT     = r"C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\0_Reference_without_e"
OUTPUT_EXCEL   = r"C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\PLOTS_5\match_report.xlsx"

# PLOTTING 
# True  = save the same 3-panel best-match PNG plots as the single-spectrum script.
#         Plots are automatically sorted into Over_Threshold and Below_Threshold folders.
# False = Excel report only; no plot files are written.
PLOT_ENABLED = True
OUTPUT_PLOT_DIR = os.path.join(os.path.dirname(OUTPUT_EXCEL), "best_match_plots")
OVER_THRESHOLD_FOLDER = "Over_Threshold"
BELOW_THRESHOLD_FOLDER = "Below_Threshold"

TQDM_ENABLED = True


def _progress(iterable, desc=None, total=None, unit="it", enabled=True):
    if enabled and _TQDM_AVAILABLE:
        return tqdm(
            iterable,
            desc=desc,
            total=total,
            unit=unit,
            file=sys.stdout,
            dynamic_ncols=True,
            leave=True,
            ascii=True,
            mininterval=0.2,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
    return iterable


#Analysis window (user-selected: 500-3500)
RANGE_MIN = 500
RANGE_MAX = 3500

#Scoring (engine values) 
THRESHOLD       = 70
PEAK_TOLERANCE  = 10.0
NUM_TOP_PEAKS   = 7
PEAK_WEIGHT     = 0.40
PEARSON_WEIGHT  = 0.60

#Peak detection (engine values) 
PEAK_PROMINENCE        = 0.05
PEAK_MIN_HEIGHT        = 0.05
MIN_PEAK_SEPARATION_CM = 20.0

#Shared correlation grid (engine values) 
SHARED_GRID_SPACING_CM = 1.0
SHARED_GRID_MIN_POINTS = 200
SHARED_GRID_MAX_POINTS = 4000

#Confidence ladder (engine values) 
STRONG_MARGIN             = 10.0
CLOSE_TWIN_MARGIN         = 10.0
STRONG_CUTOFF_CAP         = 95.0
GAP_MIN                   = 10.0
CHEMISTRY_OVERRIDE_CUTOFF = 90.0
BIG_GAP_OVERRIDE_CUTOFF   = 20.0

# Optional exclude range, e.g. [(0, 0)] disables. Use [(lo, hi)] to drop peaks.
EXCLUDE_RANGES = None

# Emit extra sheets (Summary / Class_Long / File_Long) in the workbook.
DETAILED_OUTPUT = False


TIER_A_AGGREGATE = "Tier_A_all"
TIER_B_AGGREGATE = "Tier_B_all"
TIER_C_FOLDER    = "Tier_C"
LEGACY_TIER1_FOLDER = "Tier1_Strong"
LEGACY_TIER2_FOLDER = "Tier2_Moderate"
LEGACY_TIER3_FOLDER = "Tier3_Singletons"
LEGACY_TIER4_FOLDER = "Tier4_Excluded"
HIGH_CONF_NAME = "High_Confidence"
LOW_CONF_NAME  = "Low_Confidence"


rcParams["font.family"]    = "serif"
rcParams["font.serif"]     = ["Times New Roman", "DejaVu Serif"]
rcParams["font.size"]      = 16
rcParams["axes.linewidth"] = 1.1
rcParams["axes.labelsize"] = 18
rcParams["axes.titlesize"] = 18
rcParams["xtick.labelsize"] = 15
rcParams["ytick.labelsize"] = 15
rcParams["xtick.direction"] = "in"
rcParams["ytick.direction"] = "in"
rcParams["xtick.major.size"] = 6
rcParams["ytick.major.size"] = 6
rcParams["xtick.major.width"] = 1.1
rcParams["ytick.major.width"] = 1.1
rcParams["legend.frameon"] = False
rcParams["legend.fontsize"] = 14

FAMILY_DISPLAY_FOR_PLOT = {
    "PLASTIC":    "POLYMER",
    "NONPLASTIC": "NONPOLYMER",
    "UNKNOWN":    "UNKNOWN",
}


def _display_family_for_plot(f):
    if f is None:
        return "UNKNOWN"
    return FAMILY_DISPLAY_FOR_PLOT.get(str(f), str(f))


_TIER_AB_PATTERN = re.compile(r"^Tier_([AB])\.(\d+)$")


def _classify_tier_folder(folder_name):
    if folder_name == TIER_C_FOLDER:
        return ("C", "C")
    m = _TIER_AB_PATTERN.match(folder_name)
    if m:
        return (m.group(1), f"{m.group(1)}.{m.group(2)}")
    return None


def _display_material_family(family):
    raw = "" if family is None else str(family)
    n = raw.upper().replace("-", "").replace("_", "").replace(" ", "")
    if n == "PLASTIC":
        return "Polymer"
    if n == "NONPLASTIC":
        return "Non-polymers"
    return raw


def _family_from_library_folder(folder_name):
    raw = "" if folder_name is None else str(folder_name)
    n = raw.upper().replace("-", "").replace("_", "").replace(" ", "")
    if n in ("POLYMER", "POLYMERS", "PLASTIC", "PLASTICS"):
        return "PLASTIC"
    if n in ("NONPOLYMER", "NONPOLYMERS", "NONPLASTIC", "NONPLASTICS"):
        return "NONPLASTIC"
    return None


def discover_known_files(known_root):
    records = []
    if not os.path.isdir(known_root):
        raise FileNotFoundError(f"Known root not found: {known_root}")

    legacy_v61_map = {LEGACY_TIER1_FOLDER: "A", LEGACY_TIER2_FOLDER: "B",
                      LEGACY_TIER3_FOLDER: "C"}
    legacy_hl_map = {HIGH_CONF_NAME: "A", LOW_CONF_NAME: "C"}

    for top_level in sorted(os.listdir(known_root)):
        top_dir = os.path.join(known_root, top_level)
        if not os.path.isdir(top_dir):
            continue
        for class_name in sorted(os.listdir(top_dir)):
            class_dir = os.path.join(top_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            folder_family = _family_from_library_folder(top_level)

            new_abc_dirs = []
            has_aggregate = False
            for child in sorted(os.listdir(class_dir)):
                child_path = os.path.join(class_dir, child)
                if not os.path.isdir(child_path):
                    continue
                if child in (TIER_A_AGGREGATE, TIER_B_AGGREGATE):
                    has_aggregate = True
                    continue
                classified = _classify_tier_folder(child)
                if classified is not None:
                    bt, sub = classified
                    new_abc_dirs.append((bt, sub, child_path))

            legacy_v61_dirs = {
                label: os.path.join(class_dir, name)
                for name, label in legacy_v61_map.items()
                if os.path.isdir(os.path.join(class_dir, name))
            }
            legacy_hl_dirs = {
                label: os.path.join(class_dir, name)
                for name, label in legacy_hl_map.items()
                if os.path.isdir(os.path.join(class_dir, name))
            }
            has_tier4 = os.path.isdir(os.path.join(class_dir, LEGACY_TIER4_FOLDER))

            use_new = bool(new_abc_dirs) or has_aggregate
            use_v61 = (not use_new) and bool(legacy_v61_dirs)
            use_hl  = (not use_new) and (not use_v61) and bool(legacy_hl_dirs)

            is_plastic = use_new or use_v61 or use_hl or has_tier4
            material_family = folder_family or ("PLASTIC" if is_plastic else "NONPLASTIC")

            def _emit(fp, f, tier, subgroup):
                stem = f.replace("_bs+sm.parquet", "").replace(".parquet", "")
                fid = f"{material_family}__{class_name}__{subgroup}__{stem}"
                records.append({
                    "file_id": fid, "material_family": material_family,
                    "class_label": class_name, "tier": tier,
                    "tier_subgroup": subgroup, "top_level": top_level,
                    "file_name": f, "file_stem": stem, "file_path": fp,
                })

            if use_new:
                for bt, sub, tier_dir in new_abc_dirs:
                    for rroot, _, files in os.walk(tier_dir):
                        for f in files:
                            if f.endswith(".parquet"):
                                _emit(os.path.join(rroot, f), f, bt, sub)
            elif use_v61 or use_hl:
                src = legacy_v61_dirs if use_v61 else legacy_hl_dirs
                for tier_label, tier_dir in src.items():
                    for rroot, _, files in os.walk(tier_dir):
                        for f in files:
                            if f.endswith(".parquet"):
                                _emit(os.path.join(rroot, f), f, tier_label, tier_label)
            elif not is_plastic:
                # Untiered class folder (tier=None signals UNTIERED branch)
                for rroot, _, files in os.walk(class_dir):
                    for f in files:
                        if f.endswith(".parquet"):
                            stem = f.replace("_bs+sm.parquet", "").replace(".parquet", "")
                            records.append({
                                "file_id": f"{material_family}__{class_name}__UNTIERED__{stem}",
                                "material_family": material_family,
                                "class_label": class_name, "tier": None,
                                "tier_subgroup": None, "top_level": top_level,
                                "file_name": f, "file_stem": stem,
                                "file_path": os.path.join(rroot, f),
                            })

    fam_order = {"PLASTIC": 0, "NONPLASTIC": 1}
    tier_order = {"A": 0, "B": 1, "C": 2}

    # Fallback (B): one-level layout — known_root/<class>/*.parquet
    if not records:
        for class_name in sorted(os.listdir(known_root)):
            class_dir = os.path.join(known_root, class_name)
            if not os.path.isdir(class_dir):
                continue
            direct = sorted(f for f in os.listdir(class_dir)
                            if f.endswith(".parquet")
                            and os.path.isfile(os.path.join(class_dir, f)))
            for f in direct:
                stem = f.replace("_bs+sm.parquet", "").replace(".parquet", "")
                records.append({
                    "file_id": f"NONPLASTIC__{class_name}__UNTIERED__{stem}",
                    "material_family": "NONPLASTIC", "class_label": class_name,
                    "tier": None, "tier_subgroup": None, "top_level": "",
                    "file_name": f, "file_stem": stem,
                    "file_path": os.path.join(class_dir, f),
                })

    if not records:
        flat = sorted(f for f in os.listdir(known_root)
                      if f.endswith(".parquet")
                      and os.path.isfile(os.path.join(known_root, f)))
        if flat:
            class_name = os.path.basename(os.path.normpath(known_root)) or "Reference"
            for f in flat:
                stem = f.replace("_bs+sm.parquet", "").replace(".parquet", "")
                records.append({
                    "file_id": f"FLAT__{class_name}__UNTIERED__{stem}",
                    "material_family": "NONPLASTIC", "class_label": class_name,
                    "tier": None, "tier_subgroup": None, "top_level": "",
                    "file_name": f, "file_stem": stem,
                    "file_path": os.path.join(known_root, f),
                })

    return sorted(records, key=lambda x: (
        fam_order.get(x["material_family"], 99),
        x["class_label"],
        tier_order.get(x["tier"], 98) if x["tier"] is not None else 98,
        x["tier_subgroup"] if x["tier_subgroup"] is not None else "",
        x["file_name"],
    ))


def discover_unknown_files(unknown_root):
    records = []
    for rroot, _, files in os.walk(unknown_root):
        for f in files:
            if not f.endswith(".parquet"):
                continue
            stem = f.replace("_bs+sm.parquet", "").replace(".parquet", "")
            records.append({"file_id": stem, "file_name": f,
                            "file_stem": stem,
                            "file_path": os.path.join(rroot, f)})
    return sorted(records, key=lambda x: x["file_name"])


def compute_grid_points(range_min, range_max):
    span = float(range_max) - float(range_min)
    if span <= 0:
        raise ValueError("range_max must be > range_min")
    n = int(round(span / float(SHARED_GRID_SPACING_CM)))
    return max(SHARED_GRID_MIN_POINTS, min(SHARED_GRID_MAX_POINTS, n))


def build_shared_grid(range_min, range_max):
    n = compute_grid_points(range_min, range_max)
    return np.linspace(range_min, np.nextafter(range_max, range_min), n)


def interpolate_to_grid(subset_df, grid):
    if subset_df is None or len(subset_df) < 2:
        return None
    try:
        f = interp1d(subset_df["Wave"].values, subset_df["Normalized"].values,
                     kind="linear", bounds_error=False, fill_value=np.nan)
        return f(grid)
    except Exception:
        return None


def read_snr_from_parquet(df):
    if "SNR" not in df.columns or len(df["SNR"]) == 0:
        return float("nan")
    try:
        v = float(df["SNR"].iloc[0])
    except (TypeError, ValueError):
        return float("nan")
    return v if np.isfinite(v) else float("nan")


def select_representative_peaks(positions, intensities, num_peaks, min_sep_cm):
    if len(positions) == 0:
        return np.array([]), np.array([])
    order = np.argsort(intensities)[::-1]
    sel_p, sel_i = [], []
    for idx in order:
        p = float(positions[idx])
        if all(abs(p - sp) >= min_sep_cm for sp in sel_p):
            sel_p.append(p)
            sel_i.append(float(intensities[idx]))
        if len(sel_p) >= num_peaks:
            break
    sp = np.array(sel_p, dtype=float)
    si = np.array(sel_i, dtype=float)
    o = np.argsort(sp)
    return sp[o], si[o]


def find_top_peaks(df, range_min, range_max, num_peaks, exclude_ranges, grid):
    snr = read_snr_from_parquet(df)

    subset = df[(df["Wave"] >= range_min) & (df["Wave"] <= range_max)].copy()
    if len(subset) < 2:
        return None, None, np.array([]), np.array([]), 0, None, snr

    s_min, s_max = subset["Smoothed"].min(), subset["Smoothed"].max()
    subset["Normalized"] = ((subset["Smoothed"] - s_min) / (s_max - s_min)
                            if s_max != s_min else 0.0)

    interp_vals = interpolate_to_grid(subset, grid) if grid is not None else None

    # peak range == corr range here (engine passes the same min/max for both)
    sub_p = subset[(subset["Wave"] >= range_min) & (subset["Wave"] <= range_max)].copy()
    if len(sub_p) < 2:
        return subset, None, np.array([]), np.array([]), 0, interp_vals, snr

    waves = sub_p["Wave"].values
    norms = sub_p["Normalized"].values
    avg_spacing = (waves[-1] - waves[0]) / (len(waves) - 1) if len(waves) > 1 else 1.0
    distance = max(5, int(round(MIN_PEAK_SEPARATION_CM / avg_spacing)))

    peaks, _ = find_peaks(norms, distance=distance,
                          prominence=PEAK_PROMINENCE, height=PEAK_MIN_HEIGHT)
    if len(peaks) == 0:
        return subset, None, np.array([]), np.array([]), 0, interp_vals, snr

    pos = waves[peaks]
    ints = norms[peaks]
    if exclude_ranges:
        for r_min, r_max in exclude_ranges:
            keep = (pos < r_min) | (pos > r_max)
            pos, ints = pos[keep], ints[keep]
    if len(pos) == 0:
        return subset, None, np.array([]), np.array([]), 0, interp_vals, snr

    pos, ints = select_representative_peaks(pos, ints, num_peaks, MIN_PEAK_SEPARATION_CM)
    return subset, None, pos, ints, len(pos), interp_vals, snr


def interp_safe(result, G):
    iv = result[5] if result else None
    if iv is None:
        return np.zeros(G, dtype=float)
    arr = np.asarray(iv, dtype=float)
    arr[np.isnan(arr)] = 0.0
    return arr


def result_snr(result):
    if result is None or len(result) < 7:
        return float("nan")
    try:
        v = float(result[6])
    except (TypeError, ValueError):
        return float("nan")
    return v if np.isfinite(v) else float("nan")


def pearson_matrix(U_mat, K_mat):
    U_c = U_mat - U_mat.mean(axis=1, keepdims=True)
    K_c = K_mat - K_mat.mean(axis=1, keepdims=True)
    U_n = np.linalg.norm(U_c, axis=1, keepdims=True)
    K_n = np.linalg.norm(K_c, axis=1, keepdims=True)
    U_n[U_n == 0] = 1.0
    K_n[K_n == 0] = 1.0
    return np.clip((U_c / U_n) @ (K_c / K_n).T, 0.0, 1.0)


def assess_confidence(best_score, gap, best_tier, threshold,
                      second_score, second_class,
                      chemistry_override_cutoff, big_gap_override_cutoff):
    strong_cutoff = min(threshold + STRONG_MARGIN, STRONG_CUTOFF_CAP)
    close_twin_cutoff = min(threshold + CLOSE_TWIN_MARGIN, STRONG_CUTOFF_CAP - 5)

    if best_tier is None:
        has_second_u = second_score is not None and second_class is not None
        if best_score < threshold:
            return {"label": "REJECTED",
                    "note": (f"untiered reference; score {best_score:.2f}% is below "
                             f"the {threshold:.0f}% threshold"),
                    "flags": ["UNTIERED_REF"], "score_band": "REJECTED",
                    "gap_quality": "N/A", "reference_support": "UNTIERED",
                    "strong_signals_count": 0,
                    "near_twin_class": None, "near_twin_score": None}
        parts = ["untiered reference; only similarity score evaluated",
                 f"score {best_score:.2f}% (>= {threshold:.0f}% threshold)"]
        if has_second_u:
            parts.append(f"2nd-best class {second_class} at {second_score:.2f}% "
                         f"(gap {gap:.2f}%)")
        else:
            parts.append("no competing 2nd class")
        parts.append("verdict UNTIERED_MATCH (no tier-based confidence)")
        return {"label": "UNTIERED_MATCH", "note": "; ".join(parts),
                "flags": ["UNTIERED_REF"], "score_band": "N/A",
                "gap_quality": "N/A", "reference_support": "UNTIERED",
                "strong_signals_count": 0,
                "near_twin_class": None, "near_twin_score": None}

    if best_score < threshold:
        is_multi = best_tier in ("A", "B", "Tier1", "Tier2", "High")
        return {"label": "REJECTED",
                "note": f"best-class score {best_score:.2f}% is below the "
                        f"{threshold:.0f}% threshold",
                "flags": [], "score_band": "REJECTED", "gap_quality": "N/A",
                "reference_support": "MULTI_SPECTRUM" if is_multi else "SINGLETON",
                "strong_signals_count": 0,
                "near_twin_class": None, "near_twin_score": None}

    if best_score >= chemistry_override_cutoff:
        score_band = "CHEMISTRY_CERTAIN"
    elif best_score >= strong_cutoff:
        score_band = "STRONG"
    else:
        score_band = "ACCEPTABLE"

    has_second = second_score is not None and second_class is not None
    if has_second and gap >= big_gap_override_cutoff:
        gap_quality = "BIG_GAP"
    elif has_second and gap >= GAP_MIN:
        gap_quality = "CLEAN_WIN"
    elif has_second and second_score >= close_twin_cutoff:
        gap_quality = "CLOSE_TWIN"
    elif has_second:
        gap_quality = "WEAK_WIN"
    else:
        gap_quality = "NO_SECOND"

    is_multi = best_tier in ("A", "B", "Tier1", "Tier2", "High")
    reference_support = "MULTI_SPECTRUM" if is_multi else "SINGLETON"

    sig_chem = best_score >= chemistry_override_cutoff
    sig_gap = (has_second and gap >= big_gap_override_cutoff) or (not has_second)
    sig_ref = is_multi
    n_strong = int(sig_chem) + int(sig_gap) + int(sig_ref)
    label = "HIGH" if n_strong >= 2 else ("MEDIUM" if n_strong == 1 else "LOW")

    flags = []
    near_twin_class = near_twin_score = None
    if sig_chem:
        flags.append("CHEMISTRY_CERTAIN")
    if has_second and sig_gap:
        flags.append("BIG_GAP")
    if not has_second:
        flags.append("NO_SECOND")
    flags.append("MULTI_SPECTRUM_REF" if is_multi else "SINGLETON_REF")
    if has_second and gap_quality == "CLOSE_TWIN":
        flags.append("CLOSE_TWIN")
        near_twin_class, near_twin_score = second_class, second_score
    elif has_second and gap_quality == "WEAK_WIN":
        flags.append("WEAK_GAP")

    parts = [f"score {best_score:.2f}% "
             f"(1 point if >= {chemistry_override_cutoff:.0f}%, otherwise 0) - {int(sig_chem)}"]
    if not has_second:
        parts.append(f"no competing 2nd class "
                     f"(1 point by default when no 2nd class exists) - {int(sig_gap)}")
    else:
        parts.append(f"gap to 2nd-best class {gap:.2f}% "
                     f"(2nd: {second_class} at {second_score:.2f}%) "
                     f"(1 point if >= {big_gap_override_cutoff:.0f}%, otherwise 0) - {int(sig_gap)}")
    parts.append(f"reference is Tier {best_tier} "
                 f"(1 point if Tier A or Tier B, otherwise 0) - {int(sig_ref)}")
    parts.append(f"total {n_strong}/3 - {label}")

    return {"label": label, "note": "; ".join(parts), "flags": flags,
            "score_band": score_band, "gap_quality": gap_quality,
            "reference_support": reference_support,
            "strong_signals_count": n_strong,
            "near_twin_class": near_twin_class, "near_twin_score": near_twin_score}


def _format_snr(value):
    if value is None:
        return "--"
    try:
        return f"{value:.1f}" if np.isfinite(value) else "--"
    except Exception:
        return "--"


def _safe_plot_filename(name):
    safe = re.sub(r'[\\/:*?"<>|]', "_", str(name)).strip()
    return safe or "unknown"


def _annotate_peaks_staggered(ax, positions, intensities, color,
                              max_peaks_to_annotate):
    if len(positions) == 0 or len(positions) > max_peaks_to_annotate:
        return
    order = np.argsort(positions)
    for i, idx in enumerate(order):
        p = positions[idx]
        y = intensities[idx]
        dy, va = (12, "bottom") if i % 2 == 0 else (-14, "top")
        ax.annotate(f"{p:.1f}", xy=(p, y), xytext=(0, dy),
                    textcoords="offset points",
                    ha="center", va=va, fontsize=12,
                    color=color, fontweight="bold", clip_on=True)


def plot_best_match(unknown_result, known_result,
                    unknown_name, best_family, best_class, best_file_name,
                    best_tier, similarity_score,
                    peak_tolerance, num_top_peaks,
                    corr_range_min, corr_range_max,
                    peak_weight, pearson_weight,
                    threshold, output_path,
                    confidence_label, confidence_note,
                    second_class, second_family, second_score, gap,
                    ambiguity_flags, near_twin_class, near_twin_score,
                    strong_signals_count, reference_support,
                    unknown_snr, known_snr,
                    exclude_ranges, best_tier_subgroup):

    unknown_subset = unknown_result[0]
    unknown_pos    = unknown_result[2]
    unknown_int    = unknown_result[3]
    known_subset   = known_result[0]
    known_pos      = known_result[2]
    known_int      = known_result[3]

    if unknown_subset is None or known_subset is None:
        print("Cannot plot: missing spectrum subset.")
        return

    # Peak / Pearson sub-scores recomputed for the bar panel.
    if len(unknown_pos) > 0 and len(known_pos) > 0:
        d = np.abs(unknown_pos[:, None] - known_pos)
        m = int(np.sum(np.any(d <= peak_tolerance, axis=1)))
        denom = max(len(unknown_pos), len(known_pos), m)
        peak_score = (m / denom) * 100 if denom > 0 else 0.0
    else:
        peak_score = 0.0

    xmin = max(unknown_subset["Wave"].min(), known_subset["Wave"].min())
    xmax = min(unknown_subset["Wave"].max(), known_subset["Wave"].max())
    pearson_score = 0.0
    if xmax > xmin:
        try:
            f1 = interp1d(unknown_subset["Wave"], unknown_subset["Normalized"],
                          kind="linear", bounds_error=True)
            f2 = interp1d(known_subset["Wave"], known_subset["Normalized"],
                          kind="linear", bounds_error=True)
            x_c = np.linspace(xmin, np.nextafter(xmax, xmin), 1000)
            corr, _ = pearsonr(f1(x_c), f2(x_c))
            pearson_score = max(corr, 0.0) * 100
        except Exception:
            pearson_score = 0.0

    is_untiered = (best_tier is None) or (reference_support == "UNTIERED")


    fig = plt.figure(figsize=(16.5, 15.0))
    gs = GridSpec(nrows=3, ncols=1, figure=fig,
                  height_ratios=[4.0, 2.4, 2.6],
                  hspace=0.55,
                  left=0.075, right=0.97, top=0.91, bottom=0.06)
    ax1     = fig.add_subplot(gs[0, 0])
    ax_info = fig.add_subplot(gs[1, 0])
    ax2     = fig.add_subplot(gs[2, 0])


    ax1.plot(unknown_subset["Wave"], unknown_subset["Normalized"],
             label=f"Unknown: {unknown_name}", lw=1.8, alpha=0.85,
             color="#1f4f8b")
    ax1.plot(known_subset["Wave"], known_subset["Normalized"],
             label=f"Best file: {best_file_name}", lw=1.8, alpha=0.85,
             color="#b03030")

    if len(unknown_pos) > 0:
        ax1.scatter(unknown_pos, unknown_int,
                    color="#1f4f8b", marker="x", s=70, lw=2.0,
                    label="Unknown peaks", zorder=5)
    if len(known_pos) > 0:
        ax1.scatter(known_pos, known_int,
                    color="#b03030", marker="o", s=55,
                    edgecolor="white", lw=0.8,
                    label="Known peaks", zorder=5)

    if exclude_ranges:
        for i, (r_min, r_max) in enumerate(exclude_ranges):
            if r_max > r_min:
                ax1.axvspan(r_min, r_max, alpha=0.15, color="grey",
                            label="Excluded region" if i == 0 else None)

    for i, p in enumerate(known_pos):
        lbl = f"±{peak_tolerance} cm$^{{-1}}$ tolerance" if i == 0 else None
        ax1.axvline(p - peak_tolerance, color="#b03030", lw=0.6, ls="--",
                    alpha=0.35, label=lbl)
        ax1.axvline(p + peak_tolerance, color="#b03030", lw=0.6, ls="--",
                    alpha=0.35)

    if is_untiered:
        tier_display = "untiered"
    elif best_tier:
        tier_display = best_tier
    else:
        tier_display = "untiered"

    family_display = _display_family_for_plot(best_family)

    ax1.set_title(
        f"{unknown_name} vs class '{best_class}'\n"
        f"Family: {family_display}  |  Tier: {tier_display}  |  "
        f"Best file: {best_file_name}  |  "
        f"Similarity: {similarity_score:.2f}%"
        + (f"  |  Confidence: {confidence_label}" if confidence_label else ""),
        pad=70, fontsize=18, fontweight="bold")
    ax1.set_xlabel("Raman shift (cm$^{-1}$)", fontsize=18, labelpad=10)
    ax1.set_ylabel("Normalised intensity", fontsize=18, labelpad=10)
    ax1.set_xlim(corr_range_min, corr_range_max)
    ax1.set_ylim(-0.05, 1.30)
    ax1.tick_params(axis="both", labelsize=15)
    ax1.grid(True, axis="both", linestyle=":", linewidth=0.6,
             color="#bbbbbb", alpha=0.6)
    ax1.legend(loc="lower center", bbox_to_anchor=(0.5, 1.005),
               ncol=5, fontsize=13, framealpha=0.0,
               handletextpad=0.5, columnspacing=1.5,
               borderaxespad=0.0)
    for spine in ("top", "right"):
        ax1.spines[spine].set_visible(False)

    _annotate_peaks_staggered(ax1, unknown_pos, unknown_int,
                              color="#1f4f8b",
                              max_peaks_to_annotate=num_top_peaks)
    _annotate_peaks_staggered(ax1, known_pos, known_int,
                              color="#b03030",
                              max_peaks_to_annotate=num_top_peaks)

    #MIDDLE PANEL
    ax_info.axis("off")
    n_u = len(unknown_result[2]) if unknown_result[2] is not None else 0
    n_k = len(known_result[2])   if known_result[2]   is not None else 0

    left_rows = [
        ("Peaks",       f"unknown={n_u}, known={n_k}"),
        ("Best class",  f"{best_class}  ({family_display})"),
        ("Best tier",   f"{tier_display}  ({reference_support or '-'})"),
        ("Best score",  f"{similarity_score:.2f}%"),
    ]
    if second_class is not None:
        sf = f" ({_display_family_for_plot(second_family)})" if second_family else ""
        left_rows.append(("2nd class", f"{second_class}{sf}  ({second_score:.2f}%)"))
        if gap is not None:
            left_rows.append(("Gap to 2nd", f"{gap:.2f}%"))
    if near_twin_class and near_twin_score is not None:
        left_rows.append(("Near-twin", f"{near_twin_class} ({near_twin_score:.2f}%)"))
    left_rows.append(("Unknown SNR", _format_snr(unknown_snr)))
    left_rows.append(("Known SNR",   _format_snr(known_snr)))
    if is_untiered:
        left_rows.append(("Strong signals", "N/A (untiered)"))
    elif strong_signals_count is not None:
        left_rows.append(("Strong signals", f"{strong_signals_count}/3"))
    if ambiguity_flags:
        left_rows.append(("Flags", ", ".join(ambiguity_flags)))

    label_w = max(len(lbl) for lbl, _ in left_rows)
    left_body = "\n".join(f"{lbl.ljust(label_w)} : {val}" for lbl, val in left_rows)

    right_body = ""
    if is_untiered:
        verdict = "UNTIERED_MATCH" if similarity_score >= threshold else "REJECTED"
        rel = ">=" if similarity_score >= threshold else "<"
        right_body = (
            "No tiers found - similarity-only comparison.\n\n"
            "Reference library has no tier metadata for this class,\n"
            "so the three-axis confidence rule does not apply.\n\n"
            "Verdict is based on the similarity score alone:\n"
            f"  score {similarity_score:.2f}% {rel} {threshold:.0f}% = {verdict}"
        )
    elif confidence_note:
        wrapped = []
        for p in [s.strip() for s in confidence_note.split(";") if s.strip()]:
            chunks = textwrap.wrap(p, width=58) or [p]
            wrapped.append(chunks[0])
            wrapped.extend("  " + c for c in chunks[1:])
        right_body = "\n".join(wrapped)

    header_y = 0.95
    ax_info.text(0.25, header_y, "Match details",
                 transform=ax_info.transAxes,
                 ha="center", va="center", fontsize=17, fontweight="bold",
                 color="#222222",
                 bbox=dict(facecolor="white", edgecolor="none", pad=4.0),
                 zorder=4)
    ax_info.add_line(Line2D([0.05, 0.45], [header_y, header_y],
                            transform=ax_info.transAxes,
                            color="#222222", linestyle=":", linewidth=1.4,
                            zorder=3))
    ax_info.text(0.05, 0.85, left_body,
                 transform=ax_info.transAxes,
                 ha="left", va="top", fontsize=14,
                 family="monospace", color="#222222",
                 linespacing=1.45, zorder=3)

    right_header = ("Confidence reasoning  (untiered: similarity-only)"
                    if is_untiered else "Confidence reasoning")
    ax_info.text(0.75, header_y, right_header,
                 transform=ax_info.transAxes,
                 ha="center", va="center", fontsize=17, fontweight="bold",
                 color="#222222",
                 bbox=dict(facecolor="white", edgecolor="none", pad=4.0),
                 zorder=4)
    ax_info.add_line(Line2D([0.55, 0.95], [header_y, header_y],
                            transform=ax_info.transAxes,
                            color="#222222", linestyle=":", linewidth=1.4,
                            zorder=3))
    ax_info.text(0.55, 0.85, right_body,
                 transform=ax_info.transAxes,
                 ha="left", va="top", fontsize=14,
                 family="monospace", color="#222222",
                 linespacing=1.45, zorder=3)

    #BOTTOM PANEL
    metrics = ["Peak match", "Full-spectrum corr.", "Total similarity"]
    scores  = [peak_score, pearson_score, similarity_score]
    weights = [peak_weight * 100, pearson_weight * 100, 100]

    bar_colors = ["#9ec5e8", "#3a78b8", "#1f3f72"]

    bars = ax2.bar(metrics, scores, color=bar_colors, alpha=0.92,
                   edgecolor="#1a2a44", linewidth=0.8)
    ax2.axhline(y=threshold, color="#b03030", linestyle="--", lw=1.4,
                alpha=0.85, label=f"{threshold}% threshold")
    ax2.legend(loc="upper right", fontsize=14, framealpha=0.9,
               edgecolor="#888888")
    for i, v in enumerate(scores):
        ax2.text(i, v + 2, f"{v:.2f}%\n(w={weights[i]:.0f}%)",
                 ha="center", va="bottom", fontsize=15)
    ax2.set_ylabel("Score (%)", fontsize=18, labelpad=10)
    ax2.set_title("Similarity metrics", pad=12, fontsize=18, fontweight="bold")
    ax2.set_ylim(0, 122)
    ax2.tick_params(axis="x", labelsize=15)
    ax2.tick_params(axis="y", labelsize=14)
    ax2.grid(axis="y", alpha=0.3, linestyle=":", linewidth=0.6)
    for spine in ("top", "right"):
        ax2.spines[spine].set_visible(False)
    _ = bars  # keep reference; no further styling

    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def analyze_folders(cfg):
    unknown_folder = cfg["unknown_folder"]
    known_root     = cfg["known_root"]
    output_file    = cfg["output_file"]
    threshold      = cfg["threshold"]
    peak_tolerance = cfg["peak_tolerance"]
    num_top_peaks  = cfg["num_top_peaks"]
    range_min      = cfg["range_min"]
    range_max      = cfg["range_max"]
    exclude_ranges = cfg["exclude_ranges"]
    detailed       = cfg["detailed_output"]
    chem_cut       = cfg["chemistry_override_cutoff"]
    biggap_cut     = cfg["big_gap_override_cutoff"]
    plot_enabled   = bool(cfg["plot_enabled"])
    plot_dir       = cfg["plot_dir"]
    tqdm_enabled   = bool(cfg["tqdm_enabled"])

    total_w = cfg["peak_weight"] + cfg["pearson_weight"]
    peak_weight = cfg["peak_weight"] / total_w
    pearson_weight = cfg["pearson_weight"] / total_w

    print("=" * 60)
    print("SCRIPT VERSION: PNG_ONLY_THRESHOLD_FOLDERS_TQDM_STDOUT")
    print(f"PLOTTING: {'ON' if plot_enabled else 'OFF'}")
    if plot_enabled:
        over_dir = os.path.join(plot_dir, OVER_THRESHOLD_FOLDER)
        below_dir = os.path.join(plot_dir, BELOW_THRESHOLD_FOLDER)
        os.makedirs(over_dir, exist_ok=True)
        os.makedirs(below_dir, exist_ok=True)
        print("Plot format        - PNG only")
        print(f"Plot base folder   - {plot_dir}")
        print(f"Over threshold     - {over_dir}")
        print(f"Below threshold    - {below_dir}")
    else:
        over_dir = below_dir = None
        print("No PNG plots will be written; Excel report only.")
    print(f"TQDM PROGRESS: {'ON' if (tqdm_enabled and _TQDM_AVAILABLE) else 'OFF'}")
    if tqdm_enabled and not _TQDM_AVAILABLE:
        print("tqdm is not installed. Install it with: pip install tqdm")
    print("=" * 60)

    print("Discovering reference library ...")
    known_records = discover_known_files(known_root)
    unknown_records = discover_unknown_files(unknown_folder)
    if not unknown_records:
        print("ERROR: no .parquet files in unknown folder"); return None
    if not known_records:
        print("ERROR: no .parquet files under reference library root"); return None

    class_labels = sorted({r["class_label"] for r in known_records})
    families = sorted({_display_material_family(r["material_family"]) for r in known_records})
    print(f"Found {len(unknown_records)} unknowns, {len(known_records)} knowns "
          f"({len(class_labels)} classes)")
    print(f"Material families: {families}")
    print(f"Analysis range: {range_min}-{range_max} cm-1 | "
          f"weights peak={peak_weight:.2f} pearson={pearson_weight:.2f} | "
          f"threshold {threshold}")

    grid = build_shared_grid(range_min, range_max)
    G = len(grid)

    #Precompute peaks/interp for every spectrum
    print("Precomputing spectra & peaks ...")
    known_pre, unknown_pre, skipped = {}, {}, []

    def _precompute(rec):
        df = pd.read_parquet(rec["file_path"])
        return find_top_peaks(df, range_min, range_max, num_top_peaks,
                              exclude_ranges, grid)

    for rec in _progress(known_records, desc="Precompute known", unit="file", enabled=tqdm_enabled):
        try:
            known_pre[rec["file_id"]] = {"meta": rec, "result": _precompute(rec)}
        except Exception as e:
            skipped.append((rec["file_path"], str(e)))
    for rec in _progress(unknown_records, desc="Precompute unknown", unit="file", enabled=tqdm_enabled):
        try:
            unknown_pre[rec["file_id"]] = {"meta": rec, "result": _precompute(rec)}
        except Exception as e:
            skipped.append((rec["file_path"], str(e)))
    if skipped:
        print(f"  {len(skipped)} file(s) skipped during precompute")

    u_ids = list(unknown_pre.keys())
    k_ids = list(known_pre.keys())
    if not u_ids or not k_ids:
        print("ERROR: nothing usable after precompute"); return None

    #Pearson matrix
    print("Computing Pearson correlation matrix ...")
    U_mat = np.vstack([
        interp_safe(unknown_pre[u]["result"], G)
        for u in _progress(u_ids, desc="Build unknown matrix", unit="unknown", enabled=tqdm_enabled)
    ])
    K_mat = np.vstack([
        interp_safe(known_pre[k]["result"], G)
        for k in _progress(k_ids, desc="Build known matrix", unit="file", enabled=tqdm_enabled)
    ])
    R = pearson_matrix(U_mat, K_mat) * 100.0

    #File-level pair scoring
    print("Scoring file-level pairs ...")
    rows = []
    score_pbar = None
    if tqdm_enabled and _TQDM_AVAILABLE:
        score_pbar = tqdm(
            total=len(u_ids) * len(k_ids),
            desc="Score file pairs",
            unit="pair",
            file=sys.stdout,
            dynamic_ncols=True,
            leave=True,
            ascii=True,
            mininterval=0.2,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
    try:
        for ui, u_id in enumerate(u_ids):
            u_result = unknown_pre[u_id]["result"]
            u_peaks = u_result[2] if (u_result is not None and u_result[2] is not None) else np.array([])
            for ki, k_id in enumerate(k_ids):
                k_meta = known_pre[k_id]["meta"]
                k_result = known_pre[k_id]["result"]
                k_peaks = k_result[2] if (k_result is not None and k_result[2] is not None) else np.array([])
                if len(u_peaks) > 0 and len(k_peaks) > 0:
                    dists = np.abs(u_peaks[:, None] - k_peaks)
                    matched = int(np.sum(np.any(dists <= peak_tolerance, axis=1)))
                    denom = max(len(u_peaks), len(k_peaks), matched)
                    peak_score = (matched / denom) * 100
                else:
                    peak_score = 0.0
                final_score = peak_weight * peak_score + pearson_weight * float(R[ui, ki])
                _tier = k_meta["tier"]
                _sub = k_meta.get("tier_subgroup", _tier)
                rows.append({
                    "Known Class": k_meta["class_label"],
                    "Material Family": _display_material_family(k_meta["material_family"]),
                    "Tier": _tier if _tier is not None else "",
                    "Tier Subgroup": _sub if _sub is not None else "",
                    "Known File ID": k_id,
                    "Known File": k_meta["file_stem"],
                    "Unknown ID": u_id,
                    "Unknown File": unknown_pre[u_id]["meta"]["file_stem"],
                    "Peak Score (%)": float(peak_score),
                    "Pearson Score (%)": float(R[ui, ki]),
                    "Similarity (%)": float(final_score),
                })
                if score_pbar is not None:
                    score_pbar.update(1)
    finally:
        if score_pbar is not None:
            score_pbar.close()
    results_df = pd.DataFrame(rows)

    print("Building per-unknown summaries ...")
    class_level_rows, summaries = [], []
    results_by_unknown = dict(tuple(results_df.groupby("Unknown ID", sort=False)))

    for u_id in _progress(u_ids, desc="Summaries and PNG plots", unit="unknown", enabled=tqdm_enabled):
        sub = results_by_unknown.get(u_id)
        if sub is None or sub.empty:
            continue

        best_row = sub.sort_values("Similarity (%)", ascending=False).iloc[0]
        best_class = best_row["Known Class"]
        best_family = best_row["Material Family"]
        _tier_raw = best_row["Tier"]
        best_tier = _tier_raw if _tier_raw not in (None, "", "nan") else None
        best_file_id = best_row["Known File ID"]
        best_file_name = best_row["Known File"]
        best_score = float(best_row["Similarity (%)"])

        class_best = (sub.sort_values("Similarity (%)", ascending=False)
                         .groupby(["Known Class", "Material Family"], as_index=False).first()
                         .sort_values("Similarity (%)", ascending=False)
                         .reset_index(drop=True))

        others = class_best[class_best["Known Class"] != best_class]
        if len(others) > 0:
            srow = others.iloc[0]
            second_class = srow["Known Class"]
            second_family = srow["Material Family"]
            second_score = float(srow["Similarity (%)"])
        else:
            second_class = second_family = None
            second_score = 0.0
        gap = best_score - second_score if second_class else best_score

        conf = assess_confidence(
            best_score=best_score, gap=gap, best_tier=best_tier,
            threshold=threshold,
            second_score=second_score if second_class else None,
            second_class=second_class,
            chemistry_override_cutoff=chem_cut,
            big_gap_override_cutoff=biggap_cut,
        )
        confidence = conf["label"]
        flag_str = ", ".join(conf["flags"]) if conf["flags"] else ""
        final_class = "" if confidence == "REJECTED" else best_class
        final_family = "" if confidence == "REJECTED" else best_family

        u_res = unknown_pre[u_id]["result"]
        u_n_peaks = u_res[4] if (u_res is not None and u_res[4] is not None) else 0
        unknown_snr = result_snr(u_res)
        known_snr = result_snr(known_pre[best_file_id]["result"]
                               if best_file_id in known_pre else None)

        summaries.append({
            "Unknown File": unknown_pre[u_id]["meta"]["file_stem"],
            "Unknown Peak Count": u_n_peaks,
            "Best Family": final_family,
            "Best Class": final_class,
            "Best Tier": best_tier,
            "Reference Support": conf["reference_support"],
            "Best File": best_file_name,
            "Best Score (%)": best_score,
            "Second Class": second_class or "",
            "Second Family": second_family or "",
            "Second Score (%)": second_score,
            "Gap (%)": gap,
            "Score Band": conf["score_band"],
            "Gap Quality": conf["gap_quality"],
            "Strong Signals Count": conf["strong_signals_count"],
            "Match Flags": flag_str,
            "Near-Twin Class": conf["near_twin_class"] or "",
            "Near-Twin Score (%)": (conf["near_twin_score"]
                                    if conf["near_twin_score"] is not None else ""),
            "Unknown SNR": unknown_snr if np.isfinite(unknown_snr) else np.nan,
            "Known SNR": known_snr if np.isfinite(known_snr) else np.nan,
            "Confidence": confidence,
            "Confidence Note": conf["note"],
        })

        if plot_enabled:
            try:
                unknown_name = unknown_pre[u_id]["meta"]["file_stem"]
                target_plot_dir = over_dir if best_score >= threshold else below_dir
                out_png = os.path.join(
                    target_plot_dir, f"{_safe_plot_filename(unknown_name)}__best_match.png"
                )
                plot_best_match(
                    unknown_result=u_res,
                    known_result=(known_pre[best_file_id]["result"]
                                  if best_file_id in known_pre else None),
                    unknown_name=unknown_name,
                    best_family=best_family,
                    best_class=best_class,
                    best_file_name=best_file_name,
                    best_tier=best_tier,
                    similarity_score=best_score,
                    peak_tolerance=peak_tolerance,
                    num_top_peaks=num_top_peaks,
                    corr_range_min=range_min,
                    corr_range_max=range_max,
                    peak_weight=peak_weight,
                    pearson_weight=pearson_weight,
                    threshold=threshold,
                    output_path=out_png,
                    confidence_label=confidence,
                    confidence_note=conf["note"],
                    second_class=second_class,
                    second_family=second_family,
                    second_score=(second_score if second_class else None),
                    gap=gap,
                    ambiguity_flags=conf["flags"],
                    near_twin_class=conf["near_twin_class"],
                    near_twin_score=conf["near_twin_score"],
                    strong_signals_count=conf["strong_signals_count"],
                    reference_support=conf["reference_support"],
                    unknown_snr=unknown_snr,
                    known_snr=known_snr,
                    exclude_ranges=exclude_ranges,
                    best_tier_subgroup=(best_row.get("Tier Subgroup", best_tier)
                                        if hasattr(best_row, "get") else best_tier),
                )
            except Exception as e:
                print(f"  plot skipped for {u_id}: {e}")

        for _, r in class_best.iterrows():
            class_level_rows.append({
                "Unknown File": unknown_pre[u_id]["meta"]["file_stem"],
                "Known Class": r["Known Class"],
                "Known Family": r["Material Family"],
                "Best File Inside Class": r["Known File"],
                "Best Tier Inside Class": r["Tier"],
                "Class Score (%)": float(r["Similarity (%)"]),
            })

    class_level_df = pd.DataFrame(class_level_rows)
    summary_df = pd.DataFrame(summaries)

    #Excel report
    print("Writing Excel report ...")
    _write_excel(output_file, class_level_df, summary_df, results_df,
                 threshold, detailed)

    n_match = int((summary_df["Best Score (%)"] >= threshold).sum()) if not summary_df.empty else 0
    print("=" * 60)
    print(f"Done: {len(u_ids)} unknowns processed, {n_match} matched at >= {threshold}%")
    if skipped:
        print(f"Skipped ({len(skipped)}):")
        for fp, err in skipped:
            print(f"  x {fp} - {err}")
    print(f"Report - {output_file}")
    if plot_enabled:
        print(f"PNG plots over threshold  - {over_dir}")
        print(f"PNG plots below threshold - {below_dir}")
    return summary_df


def _write_excel(output_file, class_level_df, summary_df, results_df,
                 threshold, detailed):
    pivot_class = (class_level_df.pivot(index="Known Class", columns="Unknown File",
                                        values="Class Score (%)")
                   if not class_level_df.empty else pd.DataFrame())

    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
        if not pivot_class.empty:
            column_order = list(pivot_class.columns)

            best_row_s = pd.Series(
                {r["Unknown File"]: r["Best Class"] for _, r in summary_df.iterrows()},
                name="Best Class")

            def _snr_excel(val):
                try:
                    v = float(val)
                except (TypeError, ValueError):
                    return ""
                return v if np.isfinite(v) else ""

            snr_unknown_s = pd.Series(
                {r["Unknown File"]: _snr_excel(r.get("Unknown SNR", np.nan))
                 for _, r in summary_df.iterrows()}, name="SNR Unknown")
            snr_known_s = pd.Series(
                {r["Unknown File"]: _snr_excel(r.get("Known SNR", np.nan))
                 for _, r in summary_df.iterrows()}, name="SNR Known")

            summary_rows = pd.concat([
                best_row_s.to_frame().T,
                snr_unknown_s.to_frame().T,
                snr_known_s.to_frame().T,
            ]).reindex(columns=column_order)

            final_matrix = pd.concat([pivot_class, summary_rows])
            final_matrix.to_excel(writer, sheet_name="Class_Results")

            wb = writer.book
            ws = writer.sheets["Class_Results"]
            pct_fmt = wb.add_format({"num_format": "0.00"})
            hi_fmt = wb.add_format({"num_format": "0.00", "bold": True, "bg_color": "#FFF2CC"})
            snr_fmt = wb.add_format({"num_format": "0.0", "italic": True})
            text_fmt = wb.add_format({})

            n_rows = final_matrix.shape[0]
            n_cols = final_matrix.shape[1]
            n_class_rows = n_rows - 3   # three summary rows at the bottom

            for r_excel in range(1, n_rows + 1):
                df_row = r_excel - 1
                for c in range(1, n_cols + 1):
                    val = final_matrix.iloc[df_row, c - 1]
                    if df_row < n_class_rows:
                        if pd.notna(val) and isinstance(val, (int, float)):
                            ws.write(r_excel, c, val, hi_fmt if val >= threshold else pct_fmt)
                    else:
                        label = final_matrix.index[df_row]
                        if label == "Best Class":
                            ws.write(r_excel, c, val if pd.notna(val) else "", text_fmt)
                        else:
                            if isinstance(val, (int, float)) and pd.notna(val):
                                ws.write_number(r_excel, c, float(val), snr_fmt)
                            else:
                                ws.write(r_excel, c, "", snr_fmt)

        if detailed:
            if not summary_df.empty:
                summary_df.to_excel(writer, sheet_name="Summary", index=False)
            if not class_level_df.empty:
                class_level_df.to_excel(writer, sheet_name="Class_Long", index=False)
            if not results_df.empty:
                results_df.to_excel(writer, sheet_name="File_Long", index=False)


def build_cfg(args):
    ex = EXCLUDE_RANGES
    if args.exclude_min is not None and args.exclude_max is not None \
            and args.exclude_max > args.exclude_min > 0:
        ex = [(float(args.exclude_min), float(args.exclude_max))]
    return {
        "unknown_folder": args.unknown,
        "known_root": args.known,
        "output_file": args.output,
        "threshold": args.threshold,
        "peak_tolerance": args.peak_tolerance,
        "num_top_peaks": args.num_peaks,
        "range_min": args.range_min, "range_max": args.range_max,
        "peak_weight": args.peak_weight, "pearson_weight": args.pearson_weight,
        "chemistry_override_cutoff": CHEMISTRY_OVERRIDE_CUTOFF,
        "big_gap_override_cutoff": BIG_GAP_OVERRIDE_CUTOFF,
        "exclude_ranges": ex,
        "detailed_output": args.detailed,
        "plot_enabled": args.plot_enabled,
        "plot_dir": args.plot_dir,
        "tqdm_enabled": args.tqdm_enabled,
    }


def main():
    ap = argparse.ArgumentParser(description="Step 2 Raman matching.")
    ap.add_argument("--unknown", default=UNKNOWN_FOLDER)
    ap.add_argument("--known", default=KNOWN_ROOT)
    ap.add_argument("--output", default=OUTPUT_EXCEL)
    ap.add_argument("--threshold", type=float, default=THRESHOLD)
    ap.add_argument("--peak-tolerance", type=float, default=PEAK_TOLERANCE, dest="peak_tolerance")
    ap.add_argument("--num-peaks", type=int, default=NUM_TOP_PEAKS, dest="num_peaks")
    ap.add_argument("--range-min", type=float, default=RANGE_MIN, dest="range_min")
    ap.add_argument("--range-max", type=float, default=RANGE_MAX, dest="range_max")
    ap.add_argument("--peak-weight", type=float, default=PEAK_WEIGHT, dest="peak_weight")
    ap.add_argument("--pearson-weight", type=float, default=PEARSON_WEIGHT, dest="pearson_weight")
    ap.add_argument("--exclude-min", type=float, default=None, dest="exclude_min")
    ap.add_argument("--exclude-max", type=float, default=None, dest="exclude_max")
    ap.add_argument("--detailed", action="store_true", default=DETAILED_OUTPUT)
    tqdm_group = ap.add_mutually_exclusive_group()
    tqdm_group.add_argument("--tqdm", dest="tqdm_enabled", action="store_true",
                            help="Show tqdm progress bars.")
    tqdm_group.add_argument("--no-tqdm", dest="tqdm_enabled", action="store_false",
                            help="Disable tqdm progress bars.")
    ap.set_defaults(tqdm_enabled=TQDM_ENABLED)
    plot_group = ap.add_mutually_exclusive_group()
    plot_group.add_argument("--plot", dest="plot_enabled", action="store_true",
                            help="Save best-match PNG plots for each unknown.")
    plot_group.add_argument("--no-plot", dest="plot_enabled", action="store_false",
                            help="Do not save plots; write Excel only.")
    ap.set_defaults(plot_enabled=PLOT_ENABLED)
    ap.add_argument("--plot-dir", default=OUTPUT_PLOT_DIR, dest="plot_dir",
                    help="Base folder where best-match PNG files are saved. Two subfolders are created automatically: Over_Threshold and Below_Threshold.")
    args, _ = ap.parse_known_args()
    analyze_folders(build_cfg(args))


if __name__ == "__main__":
    main()

SCRIPT VERSION: PNG_ONLY_THRESHOLD_FOLDERS_TQDM_STDOUT
PLOTTING: ON
Plot format        - PNG only
Plot base folder   - C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\PLOTS_5\best_match_plots
Over threshold     - C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\PLOTS_5\best_match_plots\Over_Threshold
Below threshold    - C:\Users\alqrifym-admin\Desktop\FINAL_REFERNCE\00006_ALL_REFERENCE\PLOTS_5\best_match_plots\Below_Threshold
TQDM PROGRESS: ON
Discovering reference library ...
Found 39 unknowns, 3081 knowns (87 classes)
Material families: ['Non-polymers', 'Polymer']
Analysis range: 500-3500 cm-1 | weights peak=0.40 pearson=0.60 | threshold 70
Precomputing spectra & peaks ...
Precompute unknown: 100%|###########################################################| 39/39 [00:00<00:00, 125.70file/s]
Computing Pearson correlation matrix ...
Build known matrix: 100%|####################################################| 3081/3081 [00:00<00:00, 146526.94file/